# Phase 2 — End-to-end neural model that exploits wallet-trajectory interactions

Phase 1 established that cross-timestep wallet-trajectory information carries genuine signal beyond per-tx features (RF + 103 traj features beats RF alone by +0.005 F1 / +0.017 PR-AUC on the temporal test split, with 6 trajectory features in the top-30 by RF importance). The honest framing: **the signal is real but RF is a poor consumer of it**. RF can split on `total_n_wallets_with_illicit > k` independently of `T_btc_vs_max_prior > r`, but it cannot natively express "high-illicit-history input wallet sending to a first-appearance output wallet" without that exact cross-feature being hand-engineered.

This notebook builds the simplest possible neural model that *can* express those cross-wallet interactions, while staying strictly within the same causality contract as Phase 1.

## Design (intentionally minimal)
1. **Backbone**: project `[T's 182 raw tx features ‖ 103 phase-1 trajectory features]` → `d_model=64`.
2. **Per-incident-wallet summary vectors** (29-dim each): mean/max of selected raw tx features over the wallet's prior txs, plus log-transformed counts (priors, illicit, licit, co-wallets, partners), fractions, decayed-illicit score, recencies, ages, burstiness, velocity, and an `is_first_appearance` flag. Produced by exactly the same `per_wallet_priors` function as phase 1 — no GNN, no frozen embedder cache, no second-stage classifier.
3. **One cross-wallet attention layer** with `T` as the query and the `W = MAX_IN + MAX_OUT = 8` per-wallet vectors as keys/values. **This is the only place wallet × wallet interactions are expressed.**
4. **Residual + MLP head**: `final = LN(tx_q + ctx)`, classifier sees `[final ‖ tx_q]`.

Total parameters: ~50K. Trained end-to-end, single stage, no pretraining, no frozen cache. Class weight 5 (less aggressive than balanced's 9.2 — phase 1 RF reports show that aggressive class weight was hurting precision).

## Why this is different from `tx_sota.ipynb`
- **No GNN**: per-wallet vectors come from direct mean-pooling of raw tx features, not from message passing through zero-input wallet nodes.
- **No frozen embedder**: end-to-end training. The wallet-vector projection learns specifically what's useful for tx classification.
- **Phase 1 trajectory features as a backbone input**: the model gets the strong hand-crafted aggregates AND the per-wallet vectors. Cross-wallet attention is *additional* expressive capacity, not the only signal source.

## Causality contract — identical to phase 1
For target tx `T` at time `t`, every input uses only data with `τ < t` (strict). Specifically:
- `T`'s 182 raw features and its `Time step` are anchored to `t` (no future).
- Phase 1 trajectory features: built via `np.searchsorted(wallet_t_arr[w], t, side="left")` — strict `τ < t` priors.
- Per-wallet summary vectors: same `per_wallet_priors` function, same strict prior cutoff.
- StandardScaler is fitted on the train slice only.
- We do NOT load `wallets_features.txt` (lifetime aggregates → leak).

## Data split — identical to phase 1
- **Train**: labelled txs with `t ≤ 34`. **Test**: labelled txs with `t ≥ 35` (paper protocol).
- Internally, we carve a random 10% slice of train as a **validation set** for best-epoch checkpointing. The TEST set is never touched during training/selection. This 10% val carve is the only deviation from phase 1's RF setup, which didn't need val.

## Comparison plan
At the end we report, on the same temporal test split:
- RF [raw 182 features only] (phase 1 baseline)
- RF [raw + 103 trajectory features] (phase 1 augmented)
- **Phase 2 neural model** (this notebook)

with F1@0.5, AUC, PR-AUC, classification report, per-timestep breakdown, and attention-weight inspection on a few example txs.


In [1]:
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from pathlib import Path
from collections import defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    classification_report, precision_recall_curve,
)

# Walk up from CWD to find the repo root (the dir that has actors_data/ and transactions_data/)
ROOT = Path.cwd()
while not (ROOT / "actors_data").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
WALLETS_DIR      = ROOT / "actors_data"
TRANSACTIONS_DIR = ROOT / "transactions_data"
assert WALLETS_DIR.exists() and TRANSACTIONS_DIR.exists(), \
    f"Could not find data dirs from {Path.cwd()} (looked up to {ROOT})"

print(f"repo root: {ROOT}")

# === Same data split as phase 1 ===
TRAIN_END     = 34       # paper split: train t<=34, test t>=35
N_TIME_STEPS  = 49
RANDOM_SEED   = 175      # match phase 1
VAL_FRACTION  = 0.10     # carve from train (random) for best-epoch selection
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# === Phase 1 trajectory engineering caps (same as phase 1) ===
MAX_INCIDENT_PER_SIDE = 32       # for trajectory aggregations (matches phase 1)
MAX_CO_WALLETS        = 150
RECENCY_SENTINEL      = N_TIME_STEPS * 2
DECAY_RATE            = 0.2

# === Phase 2-specific: per-wallet attention slots ===
W_PER_SIDE      = 4              # per-wallet attention: keep top-4 input + top-4 output
W_TOTAL         = 2 * W_PER_SIDE # = 8 wallet slots per tx
F_WALLET        = 29             # per-wallet vector dim (set in cell 3)

# === Neural model ===
D_MODEL         = 64
N_HEADS         = 4
DROPOUT         = 0.3
LR              = 1e-3
WEIGHT_DECAY    = 5e-4
EPOCHS          = 30
BATCH_SIZE      = 512
CLASS_WEIGHT    = 5.0            # less aggressive than balanced (~9.2). Phase 1 RF used balanced.

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {DEVICE}")


repo root: /Users/jarayliu/Documents/GitHub/stat175-final
device: cpu


In [2]:
print("Loading transactions...")
tx_features_df = pd.read_csv(TRANSACTIONS_DIR / "txs_features.csv")
tx_classes_df  = pd.read_csv(TRANSACTIONS_DIR / "txs_classes.csv")

tx_classes_df["label"] = tx_classes_df["class"].map({1: 1, 2: 0, 3: -1}).astype(np.int8)
tx_id_to_idx = {tid: i for i, tid in enumerate(tx_features_df["txId"].values)}
N_tx = len(tx_features_df)
tx_feat_cols = [c for c in tx_features_df.columns if c not in ("txId", "Time step")]
F_TX = len(tx_feat_cols)

merged = tx_features_df[["txId","Time step"]].merge(tx_classes_df[["txId","label"]], on="txId", how="left")
tx_time  = merged["Time step"].astype(np.int64).values
tx_label = merged["label"].fillna(-1).astype(np.int64).values
tx_X_raw = tx_features_df[tx_feat_cols].fillna(0).astype(np.float32).values
print(f"  N_tx={N_tx:,}  F_TX={F_TX}")
print(f"  labels: illicit={int((tx_label==1).sum()):,}  licit={int((tx_label==0).sum()):,}  "
      f"unknown={int((tx_label==-1).sum()):,}")

print("\nLoading actor bipartite edges (addr<->tx)...")
addr_tx_df = pd.read_csv(WALLETS_DIR / "AddrTx_edgelist.txt")
tx_addr_df = pd.read_csv(WALLETS_DIR / "TxAddr_edgelist.txt")
wallet_addrs = sorted(set(addr_tx_df["input_address"].unique()) | set(tx_addr_df["output_address"].unique()))
wallet_addr_to_idx = {a: i for i, a in enumerate(wallet_addrs)}
N_w = len(wallet_addr_to_idx)
print(f"  unique wallets: {N_w:,}")

print("\nBuilding per-wallet timelines...")
def edges_with_time(edge_df, addr_col, tx_col):
    df = edge_df[[addr_col, tx_col]].copy()
    df["w"]      = df[addr_col].map(wallet_addr_to_idx)
    df["tx"]     = df[tx_col].map(tx_id_to_idx)
    df = df.dropna(subset=["w", "tx"])
    df["w"]      = df["w"].astype(np.int64)
    df["tx"]     = df["tx"].astype(np.int64)
    df["t"]      = tx_time[df["tx"].values]
    df["tx_lab"] = tx_label[df["tx"].values]
    return df[["w","tx","t","tx_lab"]]

at = edges_with_time(addr_tx_df, "input_address",  "txId")
ta = edges_with_time(tx_addr_df, "output_address", "txId")

incident_in  = defaultdict(list)
incident_out = defaultdict(list)
for tx, w in zip(at["tx"].values,  at["w"].values):  incident_in[int(tx)].append(int(w))
for tx, w in zip(ta["tx"].values,  ta["w"].values):  incident_out[int(tx)].append(int(w))

all_edges = pd.concat([at, ta], ignore_index=True)
all_edges = all_edges.drop_duplicates(subset=["w", "tx"]).sort_values(["w", "t"]).reset_index(drop=True)
g = all_edges.groupby("w", sort=False)
wallet_t_arr   = {}
wallet_tx_arr  = {}
wallet_lab_arr = {}
for w, sub in g:
    wallet_t_arr[int(w)]   = sub["t"].values.astype(np.int64)
    wallet_tx_arr[int(w)]  = sub["tx"].values.astype(np.int64)
    wallet_lab_arr[int(w)] = sub["tx_lab"].values.astype(np.int64)

# Pre-compute "wallet has any illicit prior by t" lookup (used by 2-hop signal)
wallet_has_illicit_by = {}
for w, labs in wallet_lab_arr.items():
    illicit_mask = (labs == 1)
    if illicit_mask.any():
        wallet_has_illicit_by[w] = int(wallet_t_arr[w][illicit_mask].min())

wallet_event_count = {w: len(tarr) for w, tarr in wallet_t_arr.items()}
print(f"  wallets with timelines: {len(wallet_t_arr):,}")
print(f"  total wallet-tx incidence pairs: {len(all_edges):,}")
print(f"  wallets with any illicit history: {len(wallet_has_illicit_by):,}")


Loading transactions...
  N_tx=203,769  F_TX=182
  labels: illicit=4,545  licit=42,019  unknown=157,205

Loading actor bipartite edges (addr<->tx)...
  unique wallets: 822,942

Building per-wallet timelines...
  wallets with timelines: 822,942
  total wallet-tx incidence pairs: 1,268,260
  wallets with any illicit history: 14,266


In [3]:
# =============================================================================
#  Compute BOTH the phase-1 flat trajectory features (per-tx aggregates)
#  AND per-wallet vectors for the cross-wallet attention layer.
#  Same per_wallet_priors function as phase 1 — strict τ < t.
# =============================================================================

print("Engineering trajectory features (port of phase 1 logic, extended with per-wallet tensors)...")

# Aggregation features (same as phase 1)
agg_feat_names = ["total_BTC", "fees", "num_input_addresses", "num_output_addresses"]
agg_feat_idxs  = [tx_feat_cols.index(c) for c in agg_feat_names]
total_btc_idx  = tx_feat_cols.index("total_BTC")
F_agg          = len(agg_feat_names)

def pick_top_wallets(wlist, k):
    if len(wlist) <= k: return list(wlist)
    cnts  = np.array([wallet_event_count.get(w, 0) for w in wlist], dtype=np.int64)
    order = np.argsort(-cnts, kind="stable")
    return [wlist[i] for i in order[:k]]

def per_wallet_priors(w, t_query):
    """Same function as phase 1. Strict τ < t_query. Returns None if no priors."""
    tarr = wallet_t_arr.get(w)
    if tarr is None: return None
    cut = int(np.searchsorted(tarr, t_query, side="left"))
    if cut == 0: return None

    prior_t   = tarr[:cut]
    prior_lab = wallet_lab_arr[w][:cut]
    prior_tx  = wallet_tx_arr[w][:cut]

    illicit_mask = (prior_lab == 1)
    licit_mask   = (prior_lab == 0)
    n_illicit    = int(illicit_mask.sum())
    n_licit      = int(licit_mask.sum())
    n_labelled   = n_illicit + n_licit

    last_illicit_t = int(prior_t[illicit_mask].max()) if n_illicit > 0 else -1
    if n_illicit > 0:
        decay_w = np.exp(-DECAY_RATE * (t_query - prior_t[illicit_mask]).astype(np.float64))
        decayed_illicit_score = float(decay_w.sum())
    else:
        decayed_illicit_score = 0.0

    illicit_frac = n_illicit / max(n_labelled, 1)

    # 2-hop co-illicit
    co_wallets = set()
    for tx_i in prior_tx:
        tx_i_int = int(tx_i)
        for co_w in incident_in.get(tx_i_int,  []):
            if co_w != w: co_wallets.add(co_w)
        for co_w in incident_out.get(tx_i_int, []):
            if co_w != w: co_wallets.add(co_w)
        if len(co_wallets) >= MAX_CO_WALLETS: break
    n_co_wallets = len(co_wallets)
    n_co_illicit = sum(1 for cw in co_wallets
                       if wallet_has_illicit_by.get(cw, N_TIME_STEPS + 1) < t_query)

    # Structural
    unique_in_partners, unique_out_partners = set(), set()
    for tx_i in prior_tx:
        tx_i_int = int(tx_i)
        for co_w in incident_in.get(tx_i_int,  []):
            if co_w != w: unique_in_partners.add(co_w)
        for co_w in incident_out.get(tx_i_int, []):
            if co_w != w: unique_out_partners.add(co_w)
    n_in_partners  = len(unique_in_partners)
    n_out_partners = len(unique_out_partners)
    fan_asymmetry  = n_out_partners / max(n_in_partners, 1)

    # Temporal
    age      = int(t_query - prior_t[0])
    recency  = int(t_query - prior_t[-1])
    n_recent_5 = int(((t_query - prior_t) <= 5).sum())
    n_recent_1 = int(((t_query - prior_t) <= 1).sum())
    if cut >= 2:
        iat = np.diff(prior_t.astype(np.float64))
        iat_mean = float(iat.mean())
        iat_std  = float(iat.std())
        burstiness = float((iat_std - iat_mean) / (iat_std + iat_mean + 1e-8))
    else:
        iat_mean, iat_std, burstiness = 0.0, 0.0, 0.0
    velocity = n_recent_5 / max(cut, 1)

    feat_vals = tx_X_raw[prior_tx][:, agg_feat_idxs]   # [k, F_agg]

    return {
        "n":                int(cut),
        "n_illicit":        n_illicit,
        "n_licit":          n_licit,
        "illicit_frac":     illicit_frac,
        "last_illicit_t":   last_illicit_t,
        "decayed_illicit":  decayed_illicit_score,
        "n_co_wallets":     n_co_wallets,
        "n_co_illicit":     n_co_illicit,
        "co_illicit_frac":  n_co_illicit / max(n_co_wallets, 1),
        "n_in_partners":    n_in_partners,
        "n_out_partners":   n_out_partners,
        "fan_asymmetry":    fan_asymmetry,
        "first_seen_t":     int(prior_t[0]),
        "last_seen_t":      int(prior_t[-1]),
        "age":              age,
        "recency":          recency,
        "n_recent_5":       n_recent_5,
        "n_recent_1":       n_recent_1,
        "iat_mean":         iat_mean,
        "iat_std":          iat_std,
        "burstiness":       burstiness,
        "velocity":         velocity,
        "feat_means":       feat_vals.mean(axis=0),
        "feat_maxes":       feat_vals.max(axis=0),
    }

# Same aggregator as phase 1 (per-side flat features)
def aggregate_side(summaries, side, t_T):
    n_total = len(summaries)
    valid   = [s for s in summaries if s is not None]
    n_w_prior = len(valid)
    out = {}
    p = side
    out[f"{p}_n_wallets"]              = n_total
    out[f"{p}_n_wallets_with_prior"]   = n_w_prior
    out[f"{p}_frac_first_appearance"]  = (n_total - n_w_prior) / max(n_total, 1)

    if not valid:
        zeros_keys = [
            "n_priors_sum","n_priors_max","n_illicit_sum","n_illicit_max","n_licit_sum",
            "n_wallets_with_illicit","n_wallets_illicit_frac_gt0","n_wallets_illicit_frac_gt50",
            "illicit_frac_max","illicit_frac_mean","decayed_illicit_max","decayed_illicit_sum",
            "co_illicit_sum","co_illicit_max","co_illicit_frac_max","co_illicit_frac_mean",
            "n_co_wallets_sum","fan_asymmetry_max","fan_asymmetry_mean",
            "n_in_partners_max","n_out_partners_max","frac_single_use",
            "age_max","age_mean","n_recent_5_sum","n_recent_5_max","n_recent_1_sum",
            "velocity_max","velocity_mean","burstiness_max","burstiness_mean",
            "iat_mean_min","iat_std_max",
        ]
        for k in zeros_keys: out[f"{p}_{k}"] = 0.0
        out[f"{p}_recency_to_illicit_min"] = float(RECENCY_SENTINEL)
        out[f"{p}_recency_min"]            = float(RECENCY_SENTINEL)
        for nm in agg_feat_names:
            out[f"{p}_prior_{nm}_mean_max"] = 0.0
            out[f"{p}_prior_{nm}_max_max"]  = 0.0
        return out

    ns        = np.array([s["n"]              for s in valid], dtype=np.float64)
    nis       = np.array([s["n_illicit"]      for s in valid], dtype=np.float64)
    nls       = np.array([s["n_licit"]        for s in valid], dtype=np.float64)
    ill_frac  = np.array([s["illicit_frac"]   for s in valid], dtype=np.float64)
    dec_ill   = np.array([s["decayed_illicit"]for s in valid], dtype=np.float64)
    last_ill  = np.array([s["last_illicit_t"] for s in valid], dtype=np.int64)
    has_ill   = (last_ill >= 0)
    rec_ill   = np.where(has_ill, t_T - last_ill, RECENCY_SENTINEL).astype(np.float64)

    co_ill   = np.array([s["n_co_illicit"]    for s in valid], dtype=np.float64)
    co_n     = np.array([s["n_co_wallets"]    for s in valid], dtype=np.float64)
    co_frac  = np.array([s["co_illicit_frac"] for s in valid], dtype=np.float64)
    fan_a    = np.array([s["fan_asymmetry"]   for s in valid], dtype=np.float64)
    n_inp    = np.array([s["n_in_partners"]   for s in valid], dtype=np.float64)
    n_outp   = np.array([s["n_out_partners"]  for s in valid], dtype=np.float64)

    ages = np.array([s["age"]      for s in valid], dtype=np.float64)
    recs = np.array([s["recency"]  for s in valid], dtype=np.float64)
    nr5  = np.array([s["n_recent_5"] for s in valid], dtype=np.float64)
    nr1  = np.array([s["n_recent_1"] for s in valid], dtype=np.float64)
    vel  = np.array([s["velocity"] for s in valid], dtype=np.float64)
    bur  = np.array([s["burstiness"] for s in valid], dtype=np.float64)
    iam  = np.array([s["iat_mean"] for s in valid], dtype=np.float64)
    ias  = np.array([s["iat_std"]  for s in valid], dtype=np.float64)

    feat_means = np.stack([s["feat_means"] for s in valid], axis=0)
    feat_maxes = np.stack([s["feat_maxes"] for s in valid], axis=0)

    out[f"{p}_n_priors_sum"]                = float(ns.sum())
    out[f"{p}_n_priors_max"]                = float(ns.max())
    out[f"{p}_n_illicit_sum"]               = float(nis.sum())
    out[f"{p}_n_illicit_max"]               = float(nis.max())
    out[f"{p}_n_licit_sum"]                 = float(nls.sum())
    out[f"{p}_n_wallets_with_illicit"]      = float(has_ill.sum())
    out[f"{p}_n_wallets_illicit_frac_gt0"]  = float((ill_frac > 0.0).sum())
    out[f"{p}_n_wallets_illicit_frac_gt50"] = float((ill_frac > 0.5).sum())
    out[f"{p}_illicit_frac_max"]            = float(ill_frac.max())
    out[f"{p}_illicit_frac_mean"]           = float(ill_frac.mean())
    out[f"{p}_decayed_illicit_max"]         = float(dec_ill.max())
    out[f"{p}_decayed_illicit_sum"]         = float(dec_ill.sum())
    out[f"{p}_recency_to_illicit_min"]      = float(rec_ill.min())
    out[f"{p}_co_illicit_sum"]              = float(co_ill.sum())
    out[f"{p}_co_illicit_max"]              = float(co_ill.max())
    out[f"{p}_co_illicit_frac_max"]         = float(co_frac.max())
    out[f"{p}_co_illicit_frac_mean"]        = float(co_frac.mean())
    out[f"{p}_n_co_wallets_sum"]            = float(co_n.sum())
    out[f"{p}_fan_asymmetry_max"]           = float(fan_a.max())
    out[f"{p}_fan_asymmetry_mean"]          = float(fan_a.mean())
    out[f"{p}_n_in_partners_max"]           = float(n_inp.max())
    out[f"{p}_n_out_partners_max"]          = float(n_outp.max())
    out[f"{p}_frac_single_use"]             = sum(1 for s in valid if s["n"] == 1) / max(n_w_prior, 1)
    out[f"{p}_age_max"]                     = float(ages.max())
    out[f"{p}_age_mean"]                    = float(ages.mean())
    out[f"{p}_recency_min"]                 = float(recs.min())
    out[f"{p}_n_recent_5_sum"]              = float(nr5.sum())
    out[f"{p}_n_recent_5_max"]              = float(nr5.max())
    out[f"{p}_n_recent_1_sum"]              = float(nr1.sum())
    out[f"{p}_velocity_max"]                = float(vel.max())
    out[f"{p}_velocity_mean"]               = float(vel.mean())
    out[f"{p}_burstiness_max"]              = float(bur.max())
    out[f"{p}_burstiness_mean"]             = float(bur.mean())
    out[f"{p}_iat_mean_min"]                = float(iam.min())
    out[f"{p}_iat_std_max"]                 = float(ias.max())
    for k, nm in enumerate(agg_feat_names):
        out[f"{p}_prior_{nm}_mean_max"] = float(feat_means[:, k].max())
        out[f"{p}_prior_{nm}_max_max"]  = float(feat_maxes[:, k].max())
    return out

# ── NEW: extract a fixed-length per-wallet vector ──────────────────────
def to_wallet_vec(summary, t_T):
    """Convert per_wallet_priors output to a [F_WALLET]-dim vector for attention.
    For first-appearance wallets (summary=None), all features are zero except
    the is_first_appearance flag (last position)."""
    if summary is None:
        v = np.zeros(F_WALLET, dtype=np.float32)
        v[-1] = 1.0   # is_first_appearance
        return v
    last_ill   = summary["last_illicit_t"]
    rec_to_ill = (t_T - last_ill) if last_ill >= 0 else RECENCY_SENTINEL
    return np.array([
        np.log1p(summary["n"]),
        np.log1p(summary["n_illicit"]),
        np.log1p(summary["n_licit"]),
        summary["illicit_frac"],
        summary["decayed_illicit"],
        rec_to_ill / N_TIME_STEPS,
        np.log1p(summary["n_co_wallets"]),
        np.log1p(summary["n_co_illicit"]),
        summary["co_illicit_frac"],
        np.log1p(summary["n_in_partners"]),
        np.log1p(summary["n_out_partners"]),
        np.log1p(summary["fan_asymmetry"]),
        summary["age"]     / N_TIME_STEPS,
        summary["recency"] / N_TIME_STEPS,
        np.log1p(summary["n_recent_5"]),
        np.log1p(summary["n_recent_1"]),
        summary["iat_mean"] / N_TIME_STEPS,
        summary["iat_std"]  / N_TIME_STEPS,
        summary["burstiness"],
        summary["velocity"],
        summary["feat_means"][0],
        summary["feat_means"][1],
        summary["feat_means"][2],
        summary["feat_means"][3],
        summary["feat_maxes"][0],
        summary["feat_maxes"][1],
        summary["feat_maxes"][2],
        summary["feat_maxes"][3],
        0.0,   # is_first_appearance flag (this slot HAS priors)
    ], dtype=np.float32)
assert F_WALLET == 29, "F_WALLET in cell 1 must match length of to_wallet_vec output"

# ── Main loop ─────────────────────────────────────────────────────────
print("\nComputing trajectory + per-wallet features for all txs...")
t0 = time.time()
traj_rows      = []
per_wallet_X   = np.zeros((N_tx, W_TOTAL, F_WALLET), dtype=np.float32)
per_wallet_seg = np.zeros((N_tx, W_TOTAL), dtype=np.int64)        # 0=input, 1=output
per_wallet_pad = np.ones((N_tx,  W_TOTAL), dtype=bool)            # True = no wallet at this slot

for tx_idx in range(N_tx):
    t_T = int(tx_time[tx_idx])
    T_total_btc = float(tx_X_raw[tx_idx, total_btc_idx])

    # Trajectory aggregation: top-32 (matches phase 1)
    in_w_traj  = pick_top_wallets(incident_in.get(tx_idx,  []), MAX_INCIDENT_PER_SIDE)
    out_w_traj = pick_top_wallets(incident_out.get(tx_idx, []), MAX_INCIDENT_PER_SIDE)
    in_summ_traj  = [per_wallet_priors(w, t_T) for w in in_w_traj]
    out_summ_traj = [per_wallet_priors(w, t_T) for w in out_w_traj]

    row = {}
    row.update(aggregate_side(in_summ_traj,  "in",  t_T))
    row.update(aggregate_side(out_summ_traj, "out", t_T))

    # cross-side composites and ratios (same as phase 1)
    row["both_sides_have_illicit"] = float(
        row["in_n_wallets_with_illicit"] > 0 and row["out_n_wallets_with_illicit"] > 0)
    row["total_n_illicit_priors"]  = row["in_n_illicit_sum"] + row["out_n_illicit_sum"]
    row["total_n_wallets_with_illicit"] = (
        row["in_n_wallets_with_illicit"] + row["out_n_wallets_with_illicit"])
    row["total_co_illicit"]        = row["in_co_illicit_sum"] + row["out_co_illicit_sum"]
    row["min_recency_to_illicit"]  = min(row["in_recency_to_illicit_min"],
                                         row["out_recency_to_illicit_min"])
    row["max_illicit_frac_either_side"] = max(row["in_illicit_frac_max"],
                                              row["out_illicit_frac_max"])
    row["max_decayed_illicit_either"]   = max(row["in_decayed_illicit_max"],
                                              row["out_decayed_illicit_max"])
    row["max_co_illicit_frac_either"]   = max(row["in_co_illicit_frac_max"],
                                              row["out_co_illicit_frac_max"])
    row["total_frac_first_appearance"]  = (
        (row["in_frac_first_appearance"] * max(row["in_n_wallets"], 1) +
         row["out_frac_first_appearance"] * max(row["out_n_wallets"], 1))
        / max(row["in_n_wallets"] + row["out_n_wallets"], 1))
    max_prior_btc  = max(row["in_prior_total_BTC_max_max"],  row["out_prior_total_BTC_max_max"])
    mean_prior_btc = max(row["in_prior_total_BTC_mean_max"], row["out_prior_total_BTC_mean_max"])
    row["T_btc_vs_max_prior"]  = T_total_btc / max(max_prior_btc, 1.0)
    row["T_btc_vs_mean_prior"] = T_total_btc / max(mean_prior_btc, 1.0)
    traj_rows.append(row)

    # Per-wallet vectors for attention: top-W_PER_SIDE per side (smaller cap for the model)
    in_w_attn  = pick_top_wallets(incident_in.get(tx_idx,  []), W_PER_SIDE)
    out_w_attn = pick_top_wallets(incident_out.get(tx_idx, []), W_PER_SIDE)
    for slot, w in enumerate(in_w_attn):
        per_wallet_X[tx_idx, slot]   = to_wallet_vec(per_wallet_priors(w, t_T), t_T)
        per_wallet_seg[tx_idx, slot] = 0
        per_wallet_pad[tx_idx, slot] = False
    for slot, w in enumerate(out_w_attn):
        idx = W_PER_SIDE + slot
        per_wallet_X[tx_idx, idx]   = to_wallet_vec(per_wallet_priors(w, t_T), t_T)
        per_wallet_seg[tx_idx, idx] = 1
        per_wallet_pad[tx_idx, idx] = False

    if tx_idx > 0 and tx_idx % 25_000 == 0:
        elapsed = time.time() - t0
        eta = (N_tx - tx_idx) * elapsed / tx_idx
        print(f"  tx {tx_idx:>7,}/{N_tx:,}  ({elapsed:.0f}s elapsed, ~{eta:.0f}s remaining)")

traj_df   = pd.DataFrame(traj_rows)
traj_X    = traj_df.values.astype(np.float32)
traj_cols = list(traj_df.columns)
F_TRAJ    = traj_X.shape[1]
elapsed = time.time() - t0
print(f"\n  done in {elapsed:.1f}s")
print(f"  traj_X: shape={traj_X.shape} (per-tx flat features)")
print(f"  per_wallet_X: shape={per_wallet_X.shape} (per-wallet attention tensor)")
print(f"  pad fraction: {per_wallet_pad.mean():.3f}  (= 1 - frac of wallet slots actually filled)")


Engineering trajectory features (port of phase 1 logic, extended with per-wallet tensors)...

Computing trajectory + per-wallet features for all txs...
  tx  25,000/203,769  (2s elapsed, ~17s remaining)
  tx  50,000/203,769  (7s elapsed, ~21s remaining)
  tx  75,000/203,769  (12s elapsed, ~21s remaining)
  tx 100,000/203,769  (20s elapsed, ~20s remaining)
  tx 125,000/203,769  (26s elapsed, ~17s remaining)
  tx 150,000/203,769  (30s elapsed, ~11s remaining)
  tx 175,000/203,769  (35s elapsed, ~6s remaining)
  tx 200,000/203,769  (40s elapsed, ~1s remaining)

  done in 44.2s
  traj_X: shape=(203769, 103) (per-tx flat features)
  per_wallet_X: shape=(203769, 8, 29) (per-wallet attention tensor)
  pad fraction: 0.567  (= 1 - frac of wallet slots actually filled)


In [4]:
# ── Same temporal split as phase 1 ─────────────────────────────────
labelled        = (tx_label != -1)
train_full_mask = labelled & (tx_time <= TRAIN_END)   # paper protocol
test_mask       = labelled & (tx_time >  TRAIN_END)
train_full_idx  = np.where(train_full_mask)[0]

# Carve random val from train (10%) for early stopping. RANDOM_SEED for reproducibility.
rng = np.random.default_rng(RANDOM_SEED)
shuffled = rng.permutation(train_full_idx)
n_val   = int(round(VAL_FRACTION * len(shuffled)))
val_idx_list   = shuffled[:n_val]
train_idx_list = shuffled[n_val:]
train_mask = np.zeros_like(train_full_mask)
val_mask   = np.zeros_like(train_full_mask)
train_mask[train_idx_list] = True
val_mask[val_idx_list]     = True

print(f"train: n={int(train_mask.sum()):,}  illicit_rate={tx_label[train_mask].mean():.4f}")
print(f"val:   n={int(val_mask.sum()):,}    illicit_rate={tx_label[val_mask].mean():.4f}")
print(f"test:  n={int(test_mask.sum()):,}   illicit_rate={tx_label[test_mask].mean():.4f}")

# ── Standardise inputs (fit on TRAIN only) ─────────────────────────
print("\nFitting StandardScalers on train rows only...")
tx_scaler   = StandardScaler().fit(tx_X_raw[train_mask])
traj_scaler = StandardScaler().fit(traj_X[train_mask])
tx_X_std    = tx_scaler.transform(tx_X_raw).astype(np.float32)
traj_X_std  = traj_scaler.transform(traj_X).astype(np.float32)

# Per-wallet features: fit scaler only on non-pad TRAIN slots
train_walv  = per_wallet_X[train_mask]                  # [n_train, W, F_w]
train_pad   = per_wallet_pad[train_mask]                # [n_train, W]
flat_valid  = train_walv[~train_pad]                    # [non_pad_count, F_w]
wallet_scaler = StandardScaler().fit(flat_valid)
per_wallet_X_std = wallet_scaler.transform(per_wallet_X.reshape(-1, F_WALLET)).astype(np.float32) \
                   .reshape(per_wallet_X.shape)
# Re-zero pad slots after scaling (so the projection sees zeros, not arbitrary scaler-shifted values)
per_wallet_X_std[per_wallet_pad] = 0.0

print(f"  tx_X_std: {tx_X_std.shape}  traj_X_std: {traj_X_std.shape}  per_wallet_X_std: {per_wallet_X_std.shape}")

# ── Convert masks to index arrays for clean dataloader access ─────
train_idx_arr = np.where(train_mask)[0]
val_idx_arr   = np.where(val_mask)[0]
test_idx_arr  = np.where(test_mask)[0]

class TxDataset(Dataset):
    def __init__(self, indices):
        self.indices = np.asarray(indices, dtype=np.int64)
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        idx = int(self.indices[i])
        return {
            "tx":   torch.from_numpy(tx_X_std[idx]),               # [F_TX]
            "traj": torch.from_numpy(traj_X_std[idx]),             # [F_TRAJ]
            "wal":  torch.from_numpy(per_wallet_X_std[idx]),       # [W_TOTAL, F_WALLET]
            "seg":  torch.from_numpy(per_wallet_seg[idx]),         # [W_TOTAL]
            "pad":  torch.from_numpy(per_wallet_pad[idx]),         # [W_TOTAL]
            "y":    torch.tensor(int(tx_label[idx]), dtype=torch.long),
        }

def collate(batch):
    return {
        "tx":   torch.stack([b["tx"]   for b in batch]),
        "traj": torch.stack([b["traj"] for b in batch]),
        "wal":  torch.stack([b["wal"]  for b in batch]),
        "seg":  torch.stack([b["seg"]  for b in batch]),
        "pad":  torch.stack([b["pad"]  for b in batch]),
        "y":    torch.stack([b["y"]    for b in batch]),
    }

train_loader = DataLoader(TxDataset(train_idx_arr), batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate, num_workers=0)
val_loader   = DataLoader(TxDataset(val_idx_arr),   batch_size=BATCH_SIZE * 2, shuffle=False,
                          collate_fn=collate, num_workers=0)
test_loader  = DataLoader(TxDataset(test_idx_arr),  batch_size=BATCH_SIZE * 2, shuffle=False,
                          collate_fn=collate, num_workers=0)
print(f"  loaders: train={len(train_loader)} val={len(val_loader)} test={len(test_loader)}  batches")


train: n=26,905  illicit_rate=0.1160
val:   n=2,989    illicit_rate=0.1144
test:  n=16,670   illicit_rate=0.0650

Fitting StandardScalers on train rows only...
  tx_X_std: (203769, 182)  traj_X_std: (203769, 103)  per_wallet_X_std: (203769, 8, 29)
  loaders: train=53 val=3 test=17  batches


In [5]:
class CrossWalletAttentionTx(nn.Module):
    """End-to-end neural model:
      backbone:        proj([T raw 182  ‖  T traj 103])  ->  d_model
      wallet keys/vals: proj(per-wallet 29-dim summary) + segment_emb  ->  d_model
      cross-wallet attention (T as query)               ->  ctx
      classifier on concat([LN(tx_q + ctx), tx_q])      ->  2 logits
    """
    def __init__(self, F_tx, F_traj, F_wallet, d_model=64, n_heads=4, dropout=0.3):
        super().__init__()
        self.tx_proj     = nn.Linear(F_tx + F_traj, d_model)
        self.wallet_proj = nn.Linear(F_wallet, d_model)
        self.seg_emb     = nn.Embedding(2, d_model)   # 0=INPUT, 1=OUTPUT
        self.attn        = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm        = nn.LayerNorm(d_model)
        self.dropout     = nn.Dropout(dropout)
        self.classifier  = nn.Sequential(
            nn.Linear(2 * d_model, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, 2),
        )

    @staticmethod
    def _safe_attn(attn_module, query, kv, kv_pad):
        """Wrap MHA so all-padded rows return zero instead of NaN-from-softmax."""
        all_pad = kv_pad.all(dim=-1)                     # [B]
        safe_pad = kv_pad.clone()
        safe_pad[all_pad, 0] = False
        out, attn_w = attn_module(query, kv, kv,
                                   key_padding_mask=safe_pad,
                                   need_weights=True, average_attn_weights=True)
        out = out.squeeze(1)                              # [B, d]
        out = out.masked_fill(all_pad.unsqueeze(-1), 0.0)
        return out, attn_w                                # attn_w: [B, 1, K]

    def forward(self, tx_feats, traj_feats, wallet_summ, wallet_seg, wallet_pad, return_attn=False):
        tx_q = self.tx_proj(torch.cat([tx_feats, traj_feats], dim=-1))   # [B, d]
        tx_q = self.dropout(tx_q)
        kv   = self.wallet_proj(wallet_summ) + self.seg_emb(wallet_seg)   # [B, W, d]
        kv   = self.dropout(kv)
        ctx, attn_w = self._safe_attn(self.attn, tx_q.unsqueeze(1), kv, wallet_pad)  # [B, d]
        final = self.norm(tx_q + ctx)                                     # [B, d]
        head_in = torch.cat([final, tx_q], dim=-1)                         # [B, 2d]
        logits = self.classifier(head_in)                                  # [B, 2]
        if return_attn:
            return logits, attn_w.squeeze(1)                              # [B, W]
        return logits

print("F_TX =", F_TX, "  F_TRAJ =", F_TRAJ, "  F_WALLET =", F_WALLET, "  W_TOTAL =", W_TOTAL)
model = CrossWalletAttentionTx(F_tx=F_TX, F_traj=F_TRAJ, F_wallet=F_WALLET,
                               d_model=D_MODEL, n_heads=N_HEADS, dropout=DROPOUT).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"model params: {n_params:,}")


F_TX = 182   F_TRAJ = 103   F_WALLET = 29   W_TOTAL = 8
model params: 45,506


In [6]:
optim_ = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
class_weight = torch.tensor([1.0, CLASS_WEIGHT], dtype=torch.float, device=DEVICE)
print(f"using class_weight={class_weight.tolist()}  (less aggressive than balanced ~9.2)")

def to_dev(batch):
    return {k: v.to(DEVICE) for k, v in batch.items()}

def run_eval(loader):
    model.eval()
    ys, ps = [], []
    with torch.no_grad():
        for batch in loader:
            b = to_dev(batch)
            logits = model(b["tx"], b["traj"], b["wal"], b["seg"], b["pad"])
            ys.append(b["y"].cpu().numpy())
            ps.append(logits.softmax(-1)[:, 1].cpu().numpy())
    return np.concatenate(ys), np.concatenate(ps)

best_val_f1 = -1.0
best_state  = None
best_epoch  = -1
print(f"\nTraining {EPOCHS} epochs, batch_size={BATCH_SIZE} ...")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    running, seen = 0.0, 0
    for batch in train_loader:
        b = to_dev(batch)
        optim_.zero_grad()
        logits = model(b["tx"], b["traj"], b["wal"], b["seg"], b["pad"])
        loss = F.cross_entropy(logits, b["y"], weight=class_weight)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optim_.step()
        running += float(loss.detach()) * b["y"].size(0)
        seen    += b["y"].size(0)
    vy, vp = run_eval(val_loader)
    v_f1 = f1_score(vy, (vp >= 0.5).astype(np.int64), pos_label=1, zero_division=0)
    is_best = v_f1 > best_val_f1
    if is_best:
        best_val_f1 = v_f1
        best_state  = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch  = epoch
    marker = "  *" if is_best else ""
    print(f"epoch {epoch:2d}  train_loss={running/max(seen,1):.4f}  "
          f"val_F1@0.5={v_f1:.4f}{marker}  ({time.time()-t0:.1f}s)")

print(f"\nBest val F1: {best_val_f1:.4f} at epoch {best_epoch}")
if best_state is not None:
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})


using class_weight=[1.0, 5.0]  (less aggressive than balanced ~9.2)

Training 30 epochs, batch_size=512 ...
epoch  1  train_loss=0.3184  val_F1@0.5=0.7340  *  (1.0s)
epoch  2  train_loss=0.1980  val_F1@0.5=0.8118  *  (0.6s)
epoch  3  train_loss=0.1667  val_F1@0.5=0.8498  *  (0.6s)
epoch  4  train_loss=0.1551  val_F1@0.5=0.8722  *  (0.7s)
epoch  5  train_loss=0.1371  val_F1@0.5=0.8421  (0.6s)
epoch  6  train_loss=0.1323  val_F1@0.5=0.8621  (0.6s)
epoch  7  train_loss=0.1227  val_F1@0.5=0.8880  *  (0.6s)
epoch  8  train_loss=0.1182  val_F1@0.5=0.8933  *  (0.7s)
epoch  9  train_loss=0.1166  val_F1@0.5=0.8770  (0.6s)
epoch 10  train_loss=0.1114  val_F1@0.5=0.8765  (0.5s)
epoch 11  train_loss=0.1086  val_F1@0.5=0.8534  (0.6s)
epoch 12  train_loss=0.1039  val_F1@0.5=0.8531  (0.5s)
epoch 13  train_loss=0.1033  val_F1@0.5=0.8602  (0.5s)
epoch 14  train_loss=0.0987  val_F1@0.5=0.8531  (0.5s)
epoch 15  train_loss=0.0974  val_F1@0.5=0.9014  *  (0.6s)
epoch 16  train_loss=0.0939  val_F1@0.5=0.8782

In [7]:
# ── Phase-2 neural test eval ─────────────────────────────────────
ty, tp = run_eval(test_loader)
y_test = tx_label[test_idx_arr]
test_t_arr = tx_time[test_idx_arr]

f1_neural    = f1_score(y_test, (tp >= 0.5).astype(np.int64), pos_label=1, zero_division=0)
auc_neural   = roc_auc_score(y_test, tp)
prauc_neural = average_precision_score(y_test, tp)
print("=" * 70)
print("Phase 2 neural (cross-wallet attention) — test results")
print("=" * 70)
print(f"  F1@0.5={f1_neural:.4f}  AUC={auc_neural:.4f}  PR-AUC={prauc_neural:.4f}")
print(classification_report(y_test, (tp >= 0.5).astype(np.int64),
                            target_names=["licit","illicit"], digits=4, zero_division=0))

# ── Re-run RF baselines for direct apples-to-apples comparison ────
# Use the SAME train-full mask as phase 1 (no carved val), since RF doesn't need val.
rf_train_mask = train_full_mask
y_train_rf = tx_label[rf_train_mask]
y_test_rf  = tx_label[test_mask]

print("\n" + "=" * 70)
print("RF baselines (re-run on the same data as phase 1)")
print("=" * 70)
print("--- RF [raw 182] ---")
rf_raw = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                n_jobs=-1, random_state=RANDOM_SEED)
rf_raw.fit(tx_X_raw[rf_train_mask], y_train_rf)
yp_raw  = rf_raw.predict(tx_X_raw[test_mask])
ypr_raw = rf_raw.predict_proba(tx_X_raw[test_mask])[:, 1]
f1_raw   = f1_score(y_test_rf, yp_raw, pos_label=1, zero_division=0)
auc_raw  = roc_auc_score(y_test_rf, ypr_raw)
prauc_raw= average_precision_score(y_test_rf, ypr_raw)
print(f"  F1@0.5={f1_raw:.4f}  AUC={auc_raw:.4f}  PR-AUC={prauc_raw:.4f}")

print("--- RF [raw 182 + 103 traj] ---")
X_aug_train = np.concatenate([tx_X_raw[rf_train_mask], traj_X[rf_train_mask]], axis=1)
X_aug_test  = np.concatenate([tx_X_raw[test_mask],     traj_X[test_mask]],     axis=1)
rf_aug = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                n_jobs=-1, random_state=RANDOM_SEED)
rf_aug.fit(X_aug_train, y_train_rf)
yp_aug  = rf_aug.predict(X_aug_test)
ypr_aug = rf_aug.predict_proba(X_aug_test)[:, 1]
f1_aug   = f1_score(y_test_rf, yp_aug, pos_label=1, zero_division=0)
auc_aug  = roc_auc_score(y_test_rf, ypr_aug)
prauc_aug= average_precision_score(y_test_rf, ypr_aug)
print(f"  F1@0.5={f1_aug:.4f}  AUC={auc_aug:.4f}  PR-AUC={prauc_aug:.4f}")

# ── Per-timestep test breakdown across all three models ──────────
print("\n" + "=" * 70)
print("Per-timestep test F1 (illicit class)")
print("=" * 70)
print(f"{'t':>3}  {'n':>5}  {'illicit':>7}  "
      f"{'RF[raw]':>8}  {'RF[aug]':>8}  {'Neural':>8}  {'Δ neur-aug':>11}")
for t in range(TRAIN_END + 1, N_TIME_STEPS + 1):
    m = (test_t_arr == t)
    if not m.any(): continue
    yt = y_test[m]
    if yt.sum() == 0:
        f1_r, f1_a, f1_n, ds = float("nan"), float("nan"), float("nan"), "  N/A"
    else:
        f1_r = f1_score(yt, yp_raw[m],                    pos_label=1, zero_division=0)
        f1_a = f1_score(yt, yp_aug[m],                    pos_label=1, zero_division=0)
        f1_n = f1_score(yt, (tp[m] >= 0.5).astype(int),   pos_label=1, zero_division=0)
        ds = f"{f1_n - f1_a:+.3f}"
    print(f"{t:>3}  {int(m.sum()):>5}  {int(yt.sum()):>7}  "
          f"{f1_r:>8.4f}  {f1_a:>8.4f}  {f1_n:>8.4f}  {ds:>11}")

# ── Attention-weight inspection on a few example txs ─────────────
print("\n" + "=" * 70)
print("Attention weights on 5 illicit + 5 licit test examples")
print("=" * 70)

def show_example(idx_arr, label_str):
    if len(idx_arr) == 0: return
    sample = idx_arr[:5]
    model.eval()
    with torch.no_grad():
        b = {
            "tx":   torch.from_numpy(tx_X_std[sample]).to(DEVICE),
            "traj": torch.from_numpy(traj_X_std[sample]).to(DEVICE),
            "wal":  torch.from_numpy(per_wallet_X_std[sample]).to(DEVICE),
            "seg":  torch.from_numpy(per_wallet_seg[sample]).to(DEVICE),
            "pad":  torch.from_numpy(per_wallet_pad[sample]).to(DEVICE),
        }
        logits, attn_w = model(b["tx"], b["traj"], b["wal"], b["seg"], b["pad"], return_attn=True)
        proba = logits.softmax(-1)[:, 1].cpu().numpy()
    print(f"\n  --- {label_str} ---")
    for k, idx in enumerate(sample):
        seg = per_wallet_seg[idx]
        pad = per_wallet_pad[idx]
        weights = attn_w[k].cpu().numpy()
        labels = ["I" if s == 0 else "O" for s in seg]
        labels = [l if not p else "·" for l, p in zip(labels, pad)]
        wstr = "  ".join(f"{labels[w_]}={weights[w_]:.2f}" for w_ in range(W_TOTAL))
        print(f"  tx_idx={int(idx):>6}  t={int(tx_time[idx]):>2}  pred_prob={float(proba[k]):.3f}  attn=[{wstr}]")

illicit_test = test_idx_arr[y_test == 1]
licit_test   = test_idx_arr[y_test == 0]
show_example(illicit_test, "5 illicit test txs")
show_example(licit_test,   "5 licit test txs")

# ── Save numbers for cell 8 summary ──────────────────────────────
results = {
    "RF[raw 182]":            {"f1": f1_raw,    "auc": auc_raw,    "prauc": prauc_raw},
    "RF[raw 182 + 103 traj]": {"f1": f1_aug,    "auc": auc_aug,    "prauc": prauc_aug},
    "Phase 2 neural":         {"f1": f1_neural, "auc": auc_neural, "prauc": prauc_neural},
}


Phase 2 neural (cross-wallet attention) — test results
  F1@0.5=0.6327  AUC=0.9000  PR-AUC=0.6206
              precision    recall  f1-score   support

       licit     0.9689    0.9878    0.9783     15587
     illicit     0.7561    0.5439    0.6327      1083

    accuracy                         0.9590     16670
   macro avg     0.8625    0.7658    0.8055     16670
weighted avg     0.9551    0.9590    0.9558     16670


RF baselines (re-run on the same data as phase 1)
--- RF [raw 182] ---
  F1@0.5=0.8044  AUC=0.9269  PR-AUC=0.7927
--- RF [raw 182 + 103 traj] ---
  F1@0.5=0.8050  AUC=0.9369  PR-AUC=0.8101

Per-timestep test F1 (illicit class)
  t      n  illicit   RF[raw]   RF[aug]    Neural   Δ neur-aug
 35   1341      182    0.9805    0.9805    0.9040       -0.077
 36   1708       33    0.8814    0.9000    0.6400       -0.260
 37    498       40    0.7500    0.7692    0.6000       -0.169
 38    756      111    0.9429    0.9479    0.7755       -0.172
 39   1183       81    0.9128   

In [8]:
print("=" * 70)
print(f"{'Model':30s}  {'illicit F1':>10s}  {'AUC':>8s}  {'PR-AUC':>8s}")
print("=" * 70)
for name, r in sorted(results.items(), key=lambda kv: -kv[1]["f1"]):
    print(f"  {name:28s}  {r['f1']:>10.4f}  {r['auc']:>8.4f}  {r['prauc']:>8.4f}")

print(f"\nΔ Phase 2 vs RF[aug]:  F1={results['Phase 2 neural']['f1'] - results['RF[raw 182 + 103 traj]']['f1']:+.4f}  "
      f"PR-AUC={results['Phase 2 neural']['prauc'] - results['RF[raw 182 + 103 traj]']['prauc']:+.4f}")

print("\nNotes:")
print(f"  - Same temporal split as phase 1: train t<= {TRAIN_END}, test t > {TRAIN_END}.")
print(f"    Phase 2 carves 10% random val from train for early stopping; RF uses full train.")
print(f"  - Same causal trajectory features (103) and same per_wallet_priors function as phase 1.")
print(f"  - Per-wallet attention adds 29-dim summaries × {W_TOTAL} slots, attended over by T's projection.")
print(f"  - Class weight = {CLASS_WEIGHT} (less aggressive than balanced ~{1/(tx_label[train_full_mask] == 1).mean()/2:.1f}).")
print(f"  - StandardScaler fitted on train only; per-wallet pad slots zero'd post-scale.")
print(f"  - No GNN, no frozen embedder, end-to-end training, ~{sum(p.numel() for p in model.parameters()):,} params.")


Model                           illicit F1       AUC    PR-AUC
  RF[raw 182 + 103 traj]            0.8050    0.9369    0.8101
  RF[raw 182]                       0.8044    0.9269    0.7927
  Phase 2 neural                    0.6327    0.9000    0.6206

Δ Phase 2 vs RF[aug]:  F1=-0.1724  PR-AUC=-0.1895

Notes:
  - Same temporal split as phase 1: train t<= 34, test t > 34.
    Phase 2 carves 10% random val from train for early stopping; RF uses full train.
  - Same causal trajectory features (103) and same per_wallet_priors function as phase 1.
  - Per-wallet attention adds 29-dim summaries × 8 slots, attended over by T's projection.
  - Class weight = 5.0 (less aggressive than balanced ~4.3).
  - StandardScaler fitted on train only; per-wallet pad slots zero'd post-scale.
  - No GNN, no frozen embedder, end-to-end training, ~45,506 params.
